### 1. Environment Setup & Library Imports

In [1]:
import pandas as pd
from e_commerce_sales_analysis.db import load_table, get_engine
from e_commerce_sales_analysis.cleaning import clean_city

### Project Scope: Brazilian E-Commerce (Olist) Sales & Logistics Analysis
- **Extract**: Ingest relational e-commerce tables from MySQL using connection pooling and chunked loaders.
- **Clean & Validate**: Standardize geographic names (Portuguese diacritics/accents), resolve foreign key anomalies, and preserve valid business entities.
- **Model**: Construct a Star Schema data model (Fact and Dimension tables) to prevent row explosion and revenue double-counting.
- **Export**: Store production-ready datasets in columnar Parquet format for SQL analytics and Power BI reporting.

### Text Standardization Function: `clean_city`
Brazilian address data contains multiple inconsistencies:
- Special accented Portuguese characters (e.g., *São Paulo*, *Brasília*, *Poços de Caldas*)
- Trailing two-letter state abbreviations (e.g., *Rio de Janeiro - RJ*, *Curitiba/PR*)
- Punctuation and hyphens (*Belo-Horizonte*)
- Inconsistent casing and trailing whitespace

The `clean_city` function in `e_commerce_sales_analysis.cleaning.text` standardizes these to clean ASCII title-case strings.

### Dataset Inspection: `olist_customers`
**Table Grain:** Exactly one row per customer transaction order record (`customer_id`).

**Schema Definition:**
- `customer_id`: Unique transaction-level identifier generated per purchase.
- `customer_unique_id`: Unique identifier for the actual individual customer (persistent across repeat orders).
- `customer_zip_code_prefix`: 5-digit postal code prefix.
- `customer_city`: City name.
- `customer_state`: 2-letter state code.

**Key Observations:**
- Contains 99,441 rows with zero missing values.
- `customer_id` has 99,441 unique values, while `customer_unique_id` has 96,096 unique values.
- The difference (3,345 records) represents an account-level token gap in raw data across all orders (including canceled). Across revenue orders (excluding canceled and unavailable, and excluding the 6 corrupt orders), there are 2,888 unique repeat buyers with 3,217 repeat purchase events across 98,207 revenue orders. We retain customer_unique_id throughout the pipeline to measure true repeat purchase rates and Customer Lifetime Value (LTV).


In [2]:
customer = load_table("olist_customers", chunk_size=50_000)

In [3]:
customer_copy = customer.copy()

In [4]:
customer_copy.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,06273,osasco,SP
1,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG
2,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES
3,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG
4,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,04841,sao paulo,SP


In [5]:
customer_copy.tail()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
99436,fffecc9f79fd8c764f843e9951b11341,e5794df8573fa179a90a7b797fc4b71f,95630,parobe,RS
99437,fffeda5b6d849fbd39689bb92087f431,afbb5a642107cf6bb1ca68e863175f03,22461,rio de janeiro,RJ
99438,ffff42319e9b2d713724ae527742af25,680213db6ebd9e4f24d03280cbe10346,06754,taboao da serra,SP
99439,ffffa3172527f765de70084a7e53aae8,48fd7dec70f2b104a1d5e8c5c639102b,37130,alfenas,MG
99440,ffffe8b65bbe3087b653a978c870db99,736e6bfa0510aa5b878881a226a5fd89,06172,osasco,SP


In [6]:
customer_copy.sample(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
11393,1d485563edb59324c37c002a1f96ab32,c60a3ddb000c06807f59d3d5cbbfb59f,63290,jardim,CE
37257,603f7c0d9ae8951348f59a7f8195e29c,bad0bfdbed12449641dba5328216418c,35502,divinopolis,MG
32720,547b5b32ae43c9578237a9a8ed03baf2,3b93b770a314d48d7f698a09cab03370,09725,sao bernardo do campo,SP
62135,9fe84c7ab94da8e3d39926676b4bd6b6,26546347dd1db04a1f9576b2da076b7a,92030,canoas,RS
69873,b371ddfaecd1d84938c3f6d65f420aca,092c78e2c83099d7bdc8f0fccf3654b2,85840,ceu azul,PR


In [7]:
customer_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 11.5 MB


In [8]:
customer_copy.shape

(99441, 5)

In [9]:
# Standardize customer city names using clean_city (collapses accents, hyphens, and state suffixes)
before_cities = customer_copy["customer_city"].nunique()
customer_copy["customer_city"] = clean_city(customer_copy["customer_city"])
after_cities = customer_copy["customer_city"].nunique()
print(f"Customer cities standardized: {before_cities:,} -> {after_cities:,} unique cities ({before_cities - after_cities} variants normalized)")

Customer cities standardized: 4,119 -> 4,117 unique cities (2 variants normalized)


In [10]:
print(customer_copy['customer_id'].nunique())

99441


In [11]:
print(customer_copy['customer_unique_id'].nunique())

96096


In [12]:
print(99441 - 96096)

3345


In [13]:
sum(customer_copy['customer_id'] == customer_copy['customer_unique_id'])

0

In [14]:
customer_copy['customer_id'].duplicated().sum()

np.int64(0)

In [15]:
customer_copy.duplicated().sum()

np.int64(0)

In [16]:
customer_copy.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [17]:
# Verify cleaned customer city distribution
print("Unique customer cities:", customer_copy['customer_city'].nunique())
customer_copy[['customer_city', 'customer_state']].head()

Unique customer cities: 4117


,customer_city,customer_state
0,Osasco,SP
1,Itapecerica,MG
2,Nova Venecia,ES
3,Mendonca,MG
4,Sao Paulo,SP


### Dataset Inspection: `olist_geolocation`
**Table Grain:** One row per geographic coordinate sample (`geolocation_zip_code_prefix`, `geolocation_lat`, `geolocation_lng`).

**Schema Definition:**
- `geolocation_zip_code_prefix`: 5-digit postal code prefix.
- `geolocation_lat` & `geolocation_lng`: Geographic coordinates.
- `geolocation_city`: City name.
- `geolocation_state`: 2-letter state code.

**Key Observations:**
- Contains 1,000,163 records representing geographic coordinate samples across Brazil.
- Raw city names contain accented characters, punctuation, and state abbreviations.
- Applying `clean_city` reduces unique city representations from 8,011 down to 5,947, successfully consolidating misspelled and accented variations.
- A small dictionary of explicit city name overrides is applied to align legacy municipal names (e.g., *Embu* -> *Embu Das Artes*, *Parati* -> *Paraty*) for map visualizations.

In [18]:
geo = load_table("olist_geolocation", chunk_size=50_000)

In [19]:
geo_copy = geo.copy()

In [20]:
geo_copy.head()

,geolocation_id,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1,01037,-23.545621,-46.639292,sao paulo,SP
1,2,01046,-23.546081,-46.644820,sao paulo,SP
2,3,01046,-23.546129,-46.642951,sao paulo,SP
3,4,01041,-23.544392,-46.639499,sao paulo,SP
4,5,01035,-23.541578,-46.641607,sao paulo,SP


In [21]:
geo_copy.tail()

,geolocation_id,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1000158,1000159,99950,-28.068639,-52.010705,tapejara,RS
1000159,1000160,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,1000161,99950,-28.071855,-52.014716,tapejara,RS
1000161,1000162,99980,-28.388932,-51.846871,david canabarro,RS
1000162,1000163,99950,-28.070104,-52.018658,tapejara,RS


In [22]:
geo_copy.sample(5)

,geolocation_id,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
979448,979449,96215,-32.094347,-52.181929,rio grande,RS
586771,586772,35164,-19.466580,-42.528585,ipatinga,MG
259291,259292,11628,-23.777323,-45.558973,sao sebastiao,SP
839837,839838,79740,-22.306942,-53.818186,ivinhema,MS
852912,852913,81690,-25.544585,-49.303527,curitiba,PR


In [23]:
geo_copy.shape

(1000163, 6)

In [24]:
geo_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 6 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_id               1000163 non-null  int64  
 1   geolocation_zip_code_prefix  1000163 non-null  str    
 2   geolocation_lat              1000163 non-null  float64
 3   geolocation_lng              1000163 non-null  float64
 4   geolocation_city             1000163 non-null  str    
 5   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(3)
memory usage: 62.5 MB


In [25]:
geo_copy.isna().sum()

geolocation_id                 0
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [26]:
geo_copy['geolocation_city'].value_counts()

geolocation_city
sao paulo                     135800
rio de janeiro                 62151
belo horizonte                 27805
são paulo                      24918
curitiba                       16593
                               ...  
são josé do herval                 1
almirante tamandaré do sul         1
são valentim                       1
três arroios                       1
ciríaco                            1
Name: count, Length: 8011, dtype: int64

In [27]:
print(geo_copy['geolocation_city'].nunique())

8011


In [28]:
geo_copy["geolocation_city"] = clean_city(geo_copy["geolocation_city"])


In [29]:
print(geo_copy['geolocation_city'].nunique())

5947


In [30]:
geo_copy['geolocation_city'].str.contains(r'\d+|\W+').sum()

np.int64(536112)

In [31]:
print(min(geo_copy['geolocation_lat']), max(geo_copy['geolocation_lat']))
print(min(geo_copy['geolocation_lng']), max(geo_copy['geolocation_lng']))

-36.6053744107061 45.06593318269697
-101.46676644931476 121.10539381057764


In [32]:
# Verify geolocation city counts before applying manual overrides
print("Unique geolocation cities before overrides:", geo_copy['geolocation_city'].nunique())

Unique geolocation cities before overrides: 5947


In [33]:
city_overrides = {
    "Rio De Janeiro Rio De Janeiro": "Rio De Janeiro",
    "Embu": "Embu Das Artes",
    "Santa Barbara Doeste": "Santa Barbara D Oeste",
    "Sao Joao Do Pau Dalho": "Sao Joao Do Pau D Alho",
    "Parati": "Paraty",
    }
geo_copy["geolocation_city"] = geo_copy["geolocation_city"].replace(city_overrides)

In [34]:
geo_copy.duplicated().sum()

np.int64(0)

In [35]:
geo_copy['geolocation_city'].nunique()

5943

### Dataset Inspection: `olist_products` & Category Translations
**Table Grain:** Exactly one row per catalog product (`product_id`).

**Schema Definition:**
- `product_id`: Unique product hash.
- `product_category_name`: Portuguese category name.
- Product physical attributes: photo count, weight (grams), and dimensional sizes (length, height, width in cm).

In [36]:
products = load_table("olist_products", chunk_size=50_000)

In [37]:
products_copy = products.copy()

In [38]:
product_translation = load_table("product_category_name_translation", chunk_size=50_000)

In [39]:
product_translation_copy = product_translation.copy()

In [40]:
products_copy.shape

(32951, 9)

In [41]:
product_translation_copy.shape

(73, 2)

In [42]:
product_translation_copy.head()

,product_category_name,product_category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship


In [43]:
products_copy['product_category_name'].nunique()

73

In [44]:
product_translation_copy['product_category_name_english'].nunique()

73

In [45]:
missing_cats = products_copy[
    ~products_copy["product_category_name"].isin(
        product_translation_copy["product_category_name"]
    )
]["product_category_name"].unique().tolist()
print("Categories in products but not in translation table:", missing_cats)

Categories in products but not in translation table: [nan]


**Category Translation Audit:**
In the database source, `product_category_name_translation` contains 73 categories including `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`.
The output above is `[nan]`, confirming that all 73 named Portuguese categories have English equivalents. The only missing values are the 610 products with null category in the source dataset.

In [46]:
# Merge English translations with validation
products_copy = pd.merge(
    products_copy,
    product_translation_copy,
    how="left",
    on="product_category_name",
    validate="m:1",
)
assert len(products_copy) == 32951, f"Expected 32,951 products, got {len(products_copy)}"

# Standardize product_category_name with English translations; fill the 610 nulls as 'unknown'
products_copy["product_category_name_english"] = products_copy["product_category_name_english"].fillna("unknown")
products_copy["product_category_name"] = products_copy["product_category_name_english"]
products_copy.drop(columns=["product_category_name_english"], inplace=True)

# Impute median physical dimensions and photos
for dim_col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    if dim_col in products_copy.columns:
        products_copy[dim_col] = products_copy[dim_col].fillna(products_copy[dim_col].median())

if "product_photos_qty" in products_copy.columns:
    products_copy["product_photos_qty"] = products_copy["product_photos_qty"].fillna(
        products_copy["product_photos_qty"].median()
    )

print("Cleaned products shape:", products_copy.shape)
print("Null category count:", products_copy["product_category_name"].isna().sum())
assert products_copy["product_category_name"].isna().sum() == 0, 'Null category values remain!'

Cleaned products shape: (32951, 9)
Null category count: 0


In [47]:
products_copy.head(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumery,53.0,596.0,6.0,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,auto,56.0,752.0,4.0,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,bed_bath_table,50.0,266.0,2.0,300.0,45.0,15.0,35.0
3,000b8f95fcb9e0096488278317764d19,housewares,25.0,364.0,3.0,550.0,19.0,24.0,12.0
4,000d9be29b5207b54e86aa1b1ac54872,watches_gifts,48.0,613.0,4.0,250.0,22.0,11.0,15.0


### Dataset Inspection: `olist_sellers`
**Table Grain:** Exactly one row per active merchant (`seller_id`).

**Schema Definition:**
- `seller_id`: Unique identifier for each merchant.
- `seller_zip_code_prefix`: 5-digit postal code prefix.
- `seller_city` & `seller_state`: Merchant location.

In [48]:
sellers = load_table("olist_sellers", chunk_size=50_000)
sellers_copy = sellers.copy()

In [49]:
sellers_copy.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,09080,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,29156,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,24754,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,14405,franca,SP
4,003554e2dce176b5555353e4f3555ac8,74565,goiania,GO


In [50]:
sellers_copy['seller_city'] = clean_city(sellers_copy['seller_city'])

In [51]:
sellers_copy.duplicated().sum()

np.int64(0)

In [52]:
sellers_copy.isnull().sum().sum()

np.int64(0)

### Dataset Inspection: `olist_orders`
**Table Grain:** Exactly one row per customer order (`order_id`).

**Schema Definition:**
- `order_id`: Unique order identifier.
- `customer_id`: Transaction-level foreign key referencing `olist_customers`.
- `order_status`: Order lifecycle stage (`delivered`, `shipped`, `canceled`, `invoiced`, `processing`, `unavailable`, `created`, `approved`).

In [53]:
orders = load_table("olist_orders", chunk_size=50_000)

In [54]:
orders_copy = orders.copy()

In [55]:
orders_copy.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17


In [56]:
orders_copy['order_status'].unique()

<ArrowStringArray>
[  'delivered', 'unavailable',     'shipped',    'canceled',    'invoiced',
  'processing',    'approved',     'created']
Length: 8, dtype: str

In [57]:
orders_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


In [58]:
orders_copy.duplicated().sum()

np.int64(0)

In [59]:
orders_copy.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [60]:
orders_copy.shape

(99441, 8)

#### Data Cleaning: Corrupted Delivery Auditing & Base Orders Preservation
We audit orders for impossible state transitions:
1. `order_delivered_customer_date < order_purchase_timestamp` (delivered before purchase).
2. `order_status == 'canceled'` with an active `order_delivered_customer_date`.
All other non-delivered orders (in-flight, processing, unavailable, canceled without delivery) are preserved.

In [61]:
# 1. Identify corrupted delivery records
corrupted_delivery_mask = (
    (orders_copy['order_status'] == 'delivered') &
    orders_copy['order_delivered_customer_date'].notna() &
    (orders_copy['order_delivered_customer_date'] < orders_copy['order_purchase_timestamp'])
)
print("Orders delivered before purchase:", corrupted_delivery_mask.sum())

# 2. Identify canceled orders that have a delivery timestamp (6 records)
corrupted_canceled_mask = (
    (orders_copy['order_status'] == 'canceled') &
    orders_copy['order_delivered_customer_date'].notna()
)
print(f"Canceled orders with delivery timestamps: {corrupted_canceled_mask.sum()}")

# 3. Create 'base' cleaned orders (Requirement A): Keep ALL statuses, drop only corrupt rows
base = orders_copy[~corrupted_delivery_mask & ~corrupted_canceled_mask].copy()
assert len(base) == 99435, f"Expected 99,435 orders, got {len(base)}"
print(f"Cleaned 'base' orders preserved: {len(base):,} (all {base['order_status'].nunique()} statuses kept)")

Orders delivered before purchase: 0
Canceled orders with delivery timestamps: 6
Cleaned 'base' orders preserved: 99,435 (all 8 statuses kept)


In [62]:
# Verify order status breakdown in cleaned base orders
print("Order status distribution in base:")
print(base['order_status'].value_counts())

Order status distribution in base:
order_status
delivered      96478
shipped         1107
canceled         619
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


### Dataset Inspection: `olist_order_payments`
**Table Grain:** Exactly one row per payment method / installment sequence (`order_id`, `payment_sequential`).

**Schema Definition:**
- `order_id`: Order foreign key.
- `payment_sequential`: Sequence index (allows splitting an order across multiple vouchers or cards).
- `payment_type`: Method used (`credit_card`, `boleto`, `voucher`, `debit_card`).
- `payment_installments`: Number of monthly installments.
- `payment_value`: Monetary amount paid in this transaction.

**Analytical Grain Architecture:**
- For `fact_order_payments`, we preserve the raw granular sequence and installment counts.
- For `orders_tbl` (one row per order), we aggregate payments per `order_id` (sum of payment_value, count of payment types, max installments) to prevent row explosion.

In [63]:
order_payments = load_table("olist_order_payments", chunk_size=50_000)
orders_payments_copy = order_payments.copy()
print("Raw payments shape:", orders_payments_copy.shape)
print("Unique orders in payments:", orders_payments_copy['order_id'].nunique())

Raw payments shape: (103886, 6)
Unique orders in payments: 99440


In [64]:
# Save fact_order_payments: raw payments scoped to cleaned base orders
fact_order_payments = orders_payments_copy[orders_payments_copy['order_id'].isin(base['order_id'])].copy()
print(f"fact_order_payments shape: {fact_order_payments.shape}")
assert fact_order_payments['order_id'].nunique() <= len(base)

fact_order_payments shape: (103880, 6)


In [65]:
# Aggregate payments per order from RAW payments table (Requirement B)
payments_per_order = fact_order_payments.groupby('order_id').agg(
    payment_total=('payment_value', 'sum'),
    payment_types=('payment_type', 'nunique'),
    max_installments=('payment_installments', 'max')
).reset_index()

print("Aggregated payments shape (1 row per order):", payments_per_order.shape)
assert len(payments_per_order) == payments_per_order['order_id'].nunique()

Aggregated payments shape (1 row per order): (99434, 4)


### Dataset Inspection: `olist_order_items` & Building `sales` (Fact Table)
**Table Grain:** Exactly one row per line item purchased within an order (`order_id`, `order_item_id`).

**Schema Definition:**
- `order_id`: Order foreign key.
- `order_item_id`: Line item sequence number within the order (Item 1, Item 2...).
- `product_id`: Product foreign key.
- `seller_id`: Merchant foreign key.
- `shipping_limit_date`: Seller shipping deadline.
- `price`: Item unit price.
- `freight_value`: Shipping and freight charge.

**Analytical Handling (Requirement C & Task 2):**
- Composite primary key `(order_id, order_item_id)` uniquely identifies each line item.
- We join `products_copy` and `sellers_copy` (`validate='m:1'`) so category and merchant analytics are available at item level.
- We replace the destructive `shipping_limit_date` row filter with a nullable boolean column `is_shipping_limit_valid` (0 items dropped). Null `order_approved_at` timestamps are explicitly preserved as unknown (`<NA>`).
- We do NOT include payment_value or payment_total in this table to prevent revenue multiplication.

In [66]:
orders_items = load_table("olist_order_items", chunk_size=50_000)
orders_items_copy = orders_items.copy()
print("Raw items shape:", orders_items_copy.shape)
print("Unique orders in items:", orders_items_copy['order_id'].nunique())

Raw items shape: (112650, 7)
Unique orders in items: 98666


In [67]:
# Filter items to cleaned base orders
sales = orders_items_copy[orders_items_copy['order_id'].isin(base['order_id'])].copy()
initial_items_count = len(sales)
print(f"Sales items in base orders: {initial_items_count:,}")
assert initial_items_count == 112643, f"Expected 112,643 items, got {initial_items_count}"

# Left join products with clean English category
sales = sales.merge(products_copy, on="product_id", how="left", validate="m:1")
assert len(sales) == initial_items_count, "Row count changed after products merge!"

# Left join sellers with standardized cities
sales = sales.merge(sellers_copy, on="seller_id", how="left", validate="m:1")
assert len(sales) == initial_items_count, "Row count changed after sellers merge!"

# Add order purchase and approval timestamps from base
sales = sales.merge(
    base[["order_id", "order_purchase_timestamp", "order_approved_at"]],
    on="order_id",
    how="left",
    validate="m:1"
)
assert len(sales) == initial_items_count, "Row count changed after timestamps merge!"

# Convert timestamps to datetime
sales["shipping_limit_date"] = pd.to_datetime(sales["shipping_limit_date"])
sales["order_purchase_timestamp"] = pd.to_datetime(sales["order_purchase_timestamp"])
sales["order_approved_at"] = pd.to_datetime(sales["order_approved_at"])

# Add deliberate boolean is_shipping_limit_valid (preserves all rows)
# Explicitly handle null order_approved_at:
# - Valid (True): shipping_limit_date > order_purchase_timestamp AND shipping_limit_date > order_approved_at
# - Invalid (False): shipping_limit_date <= order_approved_at (or shipping_limit_date <= order_purchase_timestamp)
# - Unknown (<NA>): order_approved_at is null (cannot compare against missing approval timestamp)
is_valid_cond = (
    sales["order_approved_at"].notna() &
    (sales["shipping_limit_date"] > sales["order_purchase_timestamp"]) &
    (sales["shipping_limit_date"] > sales["order_approved_at"])
)
is_invalid_cond = (
    sales["order_approved_at"].notna() &
    (
        (sales["shipping_limit_date"] <= sales["order_purchase_timestamp"]) |
        (sales["shipping_limit_date"] <= sales["order_approved_at"])
    )
)

sales["is_shipping_limit_valid"] = pd.Series(index=sales.index, dtype="boolean")
sales.loc[is_valid_cond, "is_shipping_limit_valid"] = True
sales.loc[is_invalid_cond, "is_shipping_limit_valid"] = False

print("Final sales table shape:", sales.shape)
print(f"Composite key (order_id, order_item_id) is unique: {not sales.duplicated(subset=['order_id', 'order_item_id']).any()}")
print("is_shipping_limit_valid breakdown:")
print(sales['is_shipping_limit_valid'].value_counts(dropna=False))

Sales items in base orders: 112,643
Final sales table shape: (112643, 21)
Composite key (order_id, order_item_id) is unique: True
is_shipping_limit_valid breakdown:
is_shipping_limit_valid
True     112501
False       127
<NA>         15
Name: count, dtype: Int64


In [68]:
sales.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,order_purchase_timestamp,order_approved_at,is_shipping_limit_valid
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,cool_stuff,58.0,598.0,...,650.0,28.0,9.0,14.0,27277,Volta Redonda,SP,2017-09-13 08:59:02,2017-09-13 09:45:35,True
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,pet_shop,56.0,239.0,...,30000.0,50.0,30.0,40.0,03471,Sao Paulo,SP,2017-04-26 10:53:06,2017-04-26 11:05:13,True
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,furniture_decor,59.0,695.0,...,3050.0,33.0,13.0,33.0,37564,Borda Da Mata,MG,2018-01-14 14:33:31,2018-01-14 14:48:30,True


### Dataset Inspection: `olist_order_reviews`
**Table Grain:** Exactly one row per review submission (`review_id`).

**Schema Definition:**
- `review_id`: Unique review identifier.
- `order_id`: Order foreign key.
- `review_score`: Customer rating (1 to 5).
- `review_comment_title` & `review_comment_message`: Text feedback.
- `review_answer_timestamp`: Survey completion timestamp.

**Deduplication:**
A small subset of orders have multiple review entries. We sort by `review_answer_timestamp` and keep the latest response per `order_id` to establish a clean `1:1` relationship.

In [69]:
orders_reviews = load_table("olist_order_reviews", chunk_size=50_000)
orders_reviews_copy = orders_reviews.copy()
print("Raw reviews count:", len(orders_reviews_copy))

# Verify distinct orders with reviews in cleaned base orders
base_order_ids = set(base['order_id'])
raw_base_distinct_orders = orders_reviews_copy[orders_reviews_copy['order_id'].isin(base_order_ids)]['order_id'].nunique()
print(f"Distinct orders with reviews in base: {raw_base_distinct_orders:,}")

# Deduplicate reviews: keep latest review per order_id for base orders
reviews_clean = (
    orders_reviews_copy[orders_reviews_copy['order_id'].isin(base_order_ids)]
    .sort_values("review_answer_timestamp")
    .drop_duplicates(subset=["order_id"], keep="last")
    .copy()
)
print(f"Deduplicated reviews for base orders: {len(reviews_clean):,}")
assert len(reviews_clean) == raw_base_distinct_orders, "Deduplication produced mismatched distinct order count!"
assert len(reviews_clean) == reviews_clean['order_id'].nunique(), "Duplicate order_ids remain in reviews_clean!"

Raw reviews count: 99224
Distinct orders with reviews in base: 98,667
Deduplicated reviews for base orders: 98,667


### Building `orders_tbl` (Dimension Table: One Row Per Order)
**Table Grain:** Exactly one row per order (`order_id`), exactly matching `len(base)` (99,435 orders).

**Architecture (Task 1: Slimming dim_orders):**
- Start with `base` (all order statuses preserved).
- `LEFT JOIN payments_per_order` (`validate="1:1"`).
- `LEFT JOIN customer_copy[["customer_id", "customer_unique_id"]]` (`validate="m:1"`). (Customer location attributes remain in `dim_customers`).
- `LEFT JOIN reviews_clean[["order_id", "review_score", "review_creation_date", "review_answer_timestamp"]]` (`validate="1:1"`). (Review comment text remains in `dim_order_reviews`).
- Add audit flags `has_items` and `has_payment` rather than dropping orders with inner joins.
- Add `is_revenue_order` boolean flag (excludes `REVENUE_EXCLUDED_STATUSES = ['canceled', 'unavailable']`).


In [70]:
# Build slimmed orders_tbl with exact 1:1 grain
orders_tbl = base.copy()
initial_orders_len = len(orders_tbl)

# Left join payments per order
orders_tbl = orders_tbl.merge(payments_per_order, on="order_id", how="left", validate="1:1")
assert len(orders_tbl) == initial_orders_len, "Length changed after payments merge!"

# Left join customer_unique_id (customer city/state/zip reside in dim_customers)
orders_tbl = orders_tbl.merge(
    customer_copy[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="m:1"
)
assert len(orders_tbl) == initial_orders_len, "Length changed after customer merge!"

# Left join review_score and timestamps (review comment text resides in dim_order_reviews)
orders_tbl = orders_tbl.merge(
    reviews_clean[["order_id", "review_score", "review_creation_date", "review_answer_timestamp"]],
    on="order_id",
    how="left",
    validate="1:1"
)
assert len(orders_tbl) == initial_orders_len, "Length changed after reviews merge!"

# Add audit and revenue flags
items_order_ids = set(sales["order_id"])
orders_tbl["has_items"] = orders_tbl["order_id"].isin(items_order_ids)
orders_tbl["has_payment"] = orders_tbl["payment_total"].notna()
REVENUE_EXCLUDED_STATUSES = ["canceled", "unavailable"]
orders_tbl["is_revenue_order"] = ~orders_tbl["order_status"].isin(REVENUE_EXCLUDED_STATUSES)

print(f"Slimmed orders_tbl shape: {orders_tbl.shape} (rows match len(base): {len(orders_tbl) == len(base)})")
print(f"  has_items:        True={orders_tbl['has_items'].sum():,}, False={(~orders_tbl['has_items']).sum():,}")
print(f"  has_payment:      True={orders_tbl['has_payment'].sum():,}, False={(~orders_tbl['has_payment']).sum():,}")
print(f"  is_revenue_order: True={orders_tbl['is_revenue_order'].sum():,}, False={(~orders_tbl['is_revenue_order']).sum():,}")
print(f"Columns in dim_orders: {orders_tbl.columns.tolist()}")

Slimmed orders_tbl shape: (99435, 18) (rows match len(base): True)
  has_items:        True=98,660, False=775
  has_payment:      True=99,434, False=1
  is_revenue_order: True=98,207, False=1,228
Columns in dim_orders: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'payment_total', 'payment_types', 'max_installments', 'customer_unique_id', 'review_score', 'review_creation_date', 'review_answer_timestamp', 'has_items', 'has_payment', 'is_revenue_order']


In [71]:
orders_tbl.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_total,payment_types,max_installments,customer_unique_id,review_score,review_creation_date,review_answer_timestamp,has_items,has_payment,is_revenue_order
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,72.19,1.0,2.0,871766c5855e863f6eccc05f988b23cb,5.0,2017-09-21,2017-09-22 10:57:03,True,True,True
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,259.83,1.0,3.0,eb28e67c4c0b83846050ddfb8a35d051,4.0,2017-05-13,2017-05-15 11:34:13,True,True,True
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,216.87,1.0,5.0,3818d81c6709e39d06b2738a8d3a2474,5.0,2018-01-23,2018-01-23 16:06:31,True,True,True


### 8. Production Export: Star Schema Artifacts (Parquet)
We export the clean Star Schema to `data/processed/`:
- **`dim_orders.parquet`**: Slimmed order dimension (`orders_tbl`, 1 row per order, 99,435 rows, no heavy comment text).
- **`fact_sales_items.parquet`**: Item sales fact (`sales`, 1 row per line item, 112,643 rows, with `is_shipping_limit_valid`).
- **`fact_order_payments.parquet`**: Payment transactions fact (`fact_order_payments`, 1 row per sequence/installment).
- **`dim_customers.parquet`**: Standardized customer dimension (`customer_copy`).
- **`dim_products.parquet`**: Cleaned product catalog with English categories (`products_copy`).
- **`dim_sellers.parquet`**: Standardized merchant dimension (`sellers_copy`).
- **`dim_order_reviews.parquet`**: Deduplicated customer reviews with comments (`reviews_clean`).

In [72]:
from e_commerce_sales_analysis.config import DATA_PROCESSED_DIR

# Ensure processed directory exists
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Remove obsolete un-normalized or flat tables if present
obsolete_files = [
    DATA_PROCESSED_DIR / "olist_master_cleaned.parquet",
    DATA_PROCESSED_DIR / "fact_order_items.parquet",
]
for old_file in obsolete_files:
    if old_file.exists():
        old_file.unlink()
        print(f"Removed obsolete file: {old_file.name}")

# Export Star Schema Dimensional Model
orders_tbl.to_parquet(DATA_PROCESSED_DIR / "dim_orders.parquet", index=False)
sales.to_parquet(DATA_PROCESSED_DIR / "fact_sales_items.parquet", index=False)
fact_order_payments.to_parquet(DATA_PROCESSED_DIR / "fact_order_payments.parquet", index=False)
customer_copy.to_parquet(DATA_PROCESSED_DIR / "dim_customers.parquet", index=False)
products_copy.to_parquet(DATA_PROCESSED_DIR / "dim_products.parquet", index=False)
sellers_copy.to_parquet(DATA_PROCESSED_DIR / "dim_sellers.parquet", index=False)
reviews_clean.to_parquet(DATA_PROCESSED_DIR / "dim_order_reviews.parquet", index=False)

print("Successfully exported 7 Star Schema Parquet files to: data/processed")
for p_file in sorted(DATA_PROCESSED_DIR.glob("*.parquet")):
    print(f"  {p_file.name:<30} {p_file.stat().st_size:,} bytes")


Successfully exported 7 Star Schema Parquet files to: data/processed
  dim_customers.parquet          6,842,549 bytes
  dim_order_reviews.parquet      9,289,209 bytes
  dim_orders.parquet             14,797,991 bytes
  dim_products.parquet           1,361,763 bytes
  dim_sellers.parquet            134,016 bytes
  fact_order_payments.parquet    4,461,161 bytes
  fact_sales_items.parquet       9,541,886 bytes


### 9. Final Pipeline Validation Audit
Rigorous mathematical reconciliation and referential integrity assertions verifying that zero rows or revenue were lost or inflated.

In [73]:
# =====================================================================
# FINAL PIPELINE VALIDATION AUDIT (Requirements G & Tasks 1, 2, 3, 5, 8)
# =====================================================================
import numpy as np

print("=" * 70)
print("FINAL PIPELINE VALIDATION AUDIT")
print("=" * 70)

# 1. orders_tbl.order_id is unique
assert orders_tbl['order_id'].is_unique, "Assertion Error: orders_tbl order_id is not unique!"
print(f"PASS 1/10: orders_tbl.order_id is strictly unique ({len(orders_tbl):,} rows).")

# 2. (order_id, order_item_id) is unique in sales
assert not sales.duplicated(subset=['order_id', 'order_item_id']).any(), "Assertion Error: sales composite key duplicated!"
print(f"PASS 2/10: sales (order_id, order_item_id) composite key is strictly unique ({len(sales):,} rows).")

# 3. Referential integrity: check for orphaned foreign keys
sales_order_orphans = len(set(sales['order_id']) - set(orders_tbl['order_id']))
pay_order_orphans = len(set(fact_order_payments['order_id']) - set(orders_tbl['order_id']))
sales_prod_orphans = len(set(sales['product_id']) - set(products_copy['product_id']))
sales_seller_orphans = len(set(sales['seller_id']) - set(sellers_copy['seller_id']))

print("\nReferential Integrity Check (Orphan Counts):")
print(f"  sales.order_id -> dim_orders:         {sales_order_orphans} orphans")
print(f"  payments.order_id -> dim_orders:      {pay_order_orphans} orphans")
print(f"  sales.product_id -> dim_products:     {sales_prod_orphans} orphans")
print(f"  sales.seller_id -> dim_sellers:       {sales_seller_orphans} orphans")

assert sales_order_orphans == 0, "Orphan order_ids found in sales!"
assert pay_order_orphans == 0, "Orphan order_ids found in payments!"
assert sales_prod_orphans == 0, "Orphan product_ids found in sales!"
assert sales_seller_orphans == 0, "Orphan seller_ids found in sales!"
print("PASS 3/10: Zero orphaned foreign keys across all star schema tables.")

# 4. sum(sales.price) equals raw sum of price for the same base orders
raw_base_items = orders_items_copy[orders_items_copy['order_id'].isin(base['order_id'])]
raw_price_sum = raw_base_items['price'].sum()
sales_price_sum = sales['price'].sum()

print("\nRevenue Reconciliation (Price):")
print(f"  Raw Items Price for base orders: R$ {raw_price_sum:,.2f}")
print(f"  Sales Table Price sum:           R$ {sales_price_sum:,.2f}")
assert np.isclose(raw_price_sum, sales_price_sum), "Revenue mismatch in item prices!"
print("PASS 4/10: sum(sales.price) exactly matches raw sum of price.")

# 5. sum(orders_tbl.payment_total) equals raw sum of payment_value for the same base orders
raw_base_pay_sum = orders_payments_copy[orders_payments_copy['order_id'].isin(base['order_id'])]['payment_value'].sum()
orders_pay_sum = orders_tbl['payment_total'].sum()

print("\nRevenue Reconciliation (Payments):")
print(f"  Raw Payments for base orders:   R$ {raw_base_pay_sum:,.2f}")
print(f"  orders_tbl payment_total sum:   R$ {orders_pay_sum:,.2f}")
assert np.isclose(raw_base_pay_sum, orders_pay_sum), "Revenue mismatch in payments!"
print("PASS 5/10: sum(orders_tbl.payment_total) exactly matches raw payment sum.")

# Payment row count assertion
payments = fact_order_payments
raw_pay = orders_payments_copy
assert len(payments) == raw_pay["order_id"].isin(base["order_id"]).sum()
print(f"PASS: payments row count matches base orders exactly ({len(payments):,} rows).")

# 6. Order status counts before vs after
print("\nOrder Status Counts (Raw Orders vs Cleaned Base Orders):")
raw_counts = orders_copy['order_status'].value_counts()
clean_counts = orders_tbl['order_status'].value_counts()
status_audit = pd.DataFrame({'Raw Orders': raw_counts, 'Cleaned Orders': clean_counts}).fillna(0).astype(int)
status_audit['Dropped'] = status_audit['Raw Orders'] - status_audit['Cleaned Orders']
print(status_audit.to_string())

assert status_audit['Dropped'].sum() == 6, f"Expected 6 dropped orders, got {status_audit['Dropped'].sum()}"
print("PASS 6/10: Exactly 6 corrupt canceled-delivered orders dropped. All other 99,435 orders are 100% preserved!")

# 7. Task 1 assertion: dim_orders has NO review comment text columns or redundant customer geography
for text_col in ['review_comment_title', 'review_comment_message', 'review_id', 'customer_city', 'customer_state', 'customer_zip_code_prefix']:
    assert text_col not in orders_tbl.columns, f"Heavy/redundant column {text_col} found in slimmed dim_orders!"
print("PASS 7/10: dim_orders is successfully slimmed (zero review comment text or redundant customer geography).")

# 8. Task 2 assertion: is_shipping_limit_valid has NO unexplained nulls
unexplained_shipping_nulls = sales['is_shipping_limit_valid'].isna() & sales['order_approved_at'].notna()
assert unexplained_shipping_nulls.sum() == 0, "Unexplained null values found in is_shipping_limit_valid!"
assert (sales['is_shipping_limit_valid'].isna() == sales['order_approved_at'].isna()).all()
valid_cnt = (sales['is_shipping_limit_valid'] == True).sum()
invalid_cnt = (sales['is_shipping_limit_valid'] == False).sum()
unknown_cnt = sales['is_shipping_limit_valid'].isna().sum()
print(f"\nShipping Limit Date Validity Breakdown:")
print(f"  Valid (True):       {valid_cnt:,}")
print(f"  Invalid (False):    {invalid_cnt:,}")
print(f"  Unknown (<NA>):     {unknown_cnt:,} (100% accounted for by null order_approved_at)")
print("PASS 8/10: is_shipping_limit_valid contains zero unexplained nulls.")

# 9. Task 3 assertion: dim_order_reviews.order_id is strictly unique
assert reviews_clean['order_id'].is_unique, "Duplicate order_id found in reviews_clean!"
raw_base_rev_orders = orders_reviews_copy[orders_reviews_copy['order_id'].isin(base['order_id'])]['order_id'].nunique()
assert len(reviews_clean) == raw_base_rev_orders, "reviews_clean length does not match distinct orders with reviews in base!"
print(f"\nReviews Table Grain Check:")
print(f"  Distinct base orders with reviews: {raw_base_rev_orders:,}")
print(f"  reviews_clean rows:               {len(reviews_clean):,}")
print("PASS 9/10: dim_order_reviews has strictly one row per order_id (never exceeds distinct base orders).")

# 10. is_revenue_order validation: print counts by status for True/False and assert no excluded status is marked True
print("\nRevenue Order Flag Breakdown by Order Status:")
status_crosstab = pd.crosstab(orders_tbl['order_status'], orders_tbl['is_revenue_order'], margins=True, margins_name='Total')
print(status_crosstab.to_string())

for exc_st in REVENUE_EXCLUDED_STATUSES:
    assert not (orders_tbl[orders_tbl['order_status'] == exc_st]['is_revenue_order']).any(), f"Excluded status {exc_st} was marked True in is_revenue_order!"

expected_rev_count = (~orders_tbl['order_status'].isin(REVENUE_EXCLUDED_STATUSES)).sum()
assert orders_tbl['is_revenue_order'].sum() == expected_rev_count
assert orders_tbl[orders_tbl['is_revenue_order']]['order_status'].isin(REVENUE_EXCLUDED_STATUSES).sum() == 0
print(f"\nPASS 10/10: is_revenue_order verified ({expected_rev_count:,} True / {(~orders_tbl['is_revenue_order']).sum():,} False). Zero excluded statuses marked True.")
print("=" * 70)

FINAL PIPELINE VALIDATION AUDIT
PASS 1/10: orders_tbl.order_id is strictly unique (99,435 rows).
PASS 2/10: sales (order_id, order_item_id) composite key is strictly unique (112,643 rows).

Referential Integrity Check (Orphan Counts):
  sales.order_id -> dim_orders:         0 orphans
  payments.order_id -> dim_orders:      0 orphans
  sales.product_id -> dim_products:     0 orphans
  sales.seller_id -> dim_sellers:       0 orphans
PASS 3/10: Zero orphaned foreign keys across all star schema tables.

Revenue Reconciliation (Price):
  Raw Items Price for base orders: R$ 13,590,997.52
  Sales Table Price sum:           R$ 13,590,997.52
PASS 4/10: sum(sales.price) exactly matches raw sum of price.

Revenue Reconciliation (Payments):
  Raw Payments for base orders:   R$ 16,008,123.54
  orders_tbl payment_total sum:   R$ 16,008,123.54
PASS 5/10: sum(orders_tbl.payment_total) exactly matches raw payment sum.
PASS: payments row count matches base orders exactly (103,880 rows).

Order Status Co